# Sparse Retrieval from Scratch: Lexical Search vs TF-IDF vs BM25

**Goal:** implement three retrieval approaches with reusable Python functions and compare their rankings on one hypothetical corpus.

No retrieval libraries are used. This notebook uses only Python's standard library (`re`, `math`, and `collections`).

## The key distinction

- **Raw lexical search:** counts query-term matches. It does not account for how rare a term is or how long a document is.
- **TF-IDF + cosine similarity:** weights terms by document frequency, then compares query and document vectors. In this notebook, term frequency is normalized by document length.
- **BM25:** weights matching terms using IDF and a *saturating* term-frequency function, while normalizing document length relative to the corpus average.

BM25 is not simply “TF-IDF with a different name.” Its term-frequency contribution has a different shape, and it has an explicit document-length normalization term.

## 1. Create a hypothetical document collection

Imagine a product-support search system. A user searches for **`python search ranking`**.

The corpus deliberately includes:
- a short, highly relevant document;
- a long document that repeats a query term many times;
- a document containing rare query terms;
- documents with partial matches.

This makes the ranking differences easier to inspect.

In [ ]:
documents = {
    "D1": "python search ranking",
    "D2": "python python python python python python search ranking",
    "D3": "python search ranking retrieval scoring relevance indexing tokenization document corpus",
    "D4": "python tutorial for beginners",
    "D5": "search ranking retrieval",
    "D6": "database indexing query optimization",
}

query = "python search ranking"

for doc_id, text in documents.items():
    print(f"{doc_id}: {text}")

D1: python search ranking
D2: python python python python python python search ranking
D3: python search ranking retrieval scoring relevance indexing tokenization document corpus
D4: python tutorial for beginners
D5: search ranking retrieval
D6: database indexing query optimization


## 2. Shared text-processing functions

All three approaches use the same simple tokenizer so that the comparison focuses on scoring, not preprocessing. This toy tokenizer lowercases text and extracts alphanumeric words.

In [ ]:
import re
import math
from collections import Counter

def tokenize(text):
    """Lowercase text and return alphanumeric word tokens."""
    return re.findall(r"\b[a-z0-9]+\b", text.lower())

def build_tokenized_corpus(documents):
    """Return {document_id: list_of_tokens}."""
    return {doc_id: tokenize(text) for doc_id, text in documents.items()}

tokenized_docs = build_tokenized_corpus(documents)
query_tokens = tokenize(query)

print("Query tokens:", query_tokens)
print("Tokenized D2:", tokenized_docs["D2"])

Query tokens: ['python', 'search', 'ranking']
Tokenized D2: ['python', 'python', 'python', 'python', 'python', 'python', 'search', 'ranking']


## 3. Method 1 — Raw lexical search

### Scoring rule

For each document, count how many **distinct query terms** occur at least once. This is a simple lexical baseline, not a full phrase search engine.

$$
score_{lex}(q,d)=\sum_{t\in unique(q)} \mathbb{1}[t\in d]
$$

Repeated occurrences do not increase this score. This choice makes the baseline easy to explain; other lexical systems may use term frequency, phrase matching, or Boolean operators.

In [ ]:
def raw_lexical_score(query_tokens, doc_tokens):
    """Count distinct query terms that appear at least once in a document."""
    doc_vocab = set(doc_tokens)
    return sum(1 for term in set(query_tokens) if term in doc_vocab)

def rank_raw_lexical(query_tokens, tokenized_docs):
    """Return (doc_id, score), sorted from highest to lowest score."""
    scored = [
        (doc_id, raw_lexical_score(query_tokens, tokens))
        for doc_id, tokens in tokenized_docs.items()
    ]
    return sorted(scored, key=lambda item: (-item[1], item[0]))

lexical_ranking = rank_raw_lexical(query_tokens, tokenized_docs)
for doc_id, score in lexical_ranking:
    print(f"{doc_id}: {score}")

D1: 3
D2: 3
D3: 3
D5: 2
D4: 1
D6: 0


### What to notice

- D1, D2, and D3 match all three query terms, so they tie.
- D2 repeats `python` many times but gets no extra credit for those repetitions.
- The method treats common and rare terms equally.
- It does not penalize long documents.

Raw lexical search is useful as a transparent baseline, but it cannot distinguish the strength of matches very well.

## 4. Method 2 — TF-IDF from scratch

TF-IDF combines two ideas:

1. **TF (term frequency):** how prominent a term is in a particular document.
2. **IDF (inverse document frequency):** how informative a term is across the corpus.

We will use length-normalized term frequency:

$$
TF(t,d)=\frac{f(t,d)}{|d|}
$$

and smoothed inverse document frequency:

$$
IDF(t)=\log\left(\frac{N+1}{df(t)+1}\right)+1
$$

where:
- $f(t,d)$ is the count of term $t$ in document $d$;
- $|d|$ is the number of tokens in document $d$;
- $N$ is the number of documents;
- $df(t)$ is the number of documents containing term $t$.

The document and query are represented as weighted vectors. We rank them using cosine similarity.

In [ ]:
def document_frequency(tokenized_docs):
    """Count in how many documents each term appears."""
    df = Counter()
    for tokens in tokenized_docs.values():
        df.update(set(tokens))
    return df

def tfidf_idf(term, df, n_docs):
    """Smoothed IDF used in this notebook."""
    return math.log((n_docs + 1) / (df.get(term, 0) + 1)) + 1

def tfidf_vector(tokens, vocabulary, df, n_docs):
    """Build a length-normalized TF-IDF vector for a token sequence."""
    counts = Counter(tokens)
    length = len(tokens) or 1
    return {
        term: (counts[term] / length) * tfidf_idf(term, df, n_docs)
        for term in vocabulary
    }

def cosine_similarity(vector_a, vector_b):
    """Cosine similarity between sparse dictionaries represented as maps."""
    dot = sum(value * vector_b.get(term, 0.0)
              for term, value in vector_a.items())
    norm_a = math.sqrt(sum(value * value for value in vector_a.values()))
    norm_b = math.sqrt(sum(value * value for value in vector_b.values()))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)

def rank_tfidf(query_tokens, tokenized_docs):
    """Rank documents by TF-IDF cosine similarity to the query."""
    n_docs = len(tokenized_docs)
    df = document_frequency(tokenized_docs)
    vocabulary = set(query_tokens)
    for tokens in tokenized_docs.values():
        vocabulary.update(tokens)

    query_vector = tfidf_vector(query_tokens, vocabulary, df, n_docs)
    scored = []
    for doc_id, tokens in tokenized_docs.items():
        doc_vector = tfidf_vector(tokens, vocabulary, df, n_docs)
        score = cosine_similarity(query_vector, doc_vector)
        scored.append((doc_id, score))
    return sorted(scored, key=lambda item: (-item[1], item[0]))

df = document_frequency(tokenized_docs)
print("Document frequency for query terms:")
for term in query_tokens:
    print(f"  {term}: df={df[term]} of N={len(tokenized_docs)}")

tfidf_ranking = rank_tfidf(query_tokens, tokenized_docs)
print("\\nTF-IDF cosine ranking:")
for doc_id, score in tfidf_ranking:
    print(f"{doc_id}: {score:.4f}")

Document frequency for query terms:
  python: df=4 of N=6
  search: df=4 of N=6
  ranking: df=4 of N=6
\nTF-IDF cosine ranking:
D1: 1.0000
D2: 0.7493
D5: 0.5839
D3: 0.3777
D4: 0.1871
D6: 0.0000


### TF-IDF interpretation

- Terms appearing in fewer documents receive larger IDF weights.
- Term frequency matters, but in this implementation it is divided by document length.
- Cosine similarity compares the angle between the query and document vectors, not their raw vector lengths.
- A document can match all query terms and still receive a lower score if it contains many unrelated terms.

**Important:** TF-IDF has multiple valid variants. Some use raw term counts, logarithmic TF, or different IDF formulas. Results depend on these choices. We define the variant explicitly so the experiment is reproducible.

## 5. Method 3 — BM25 from scratch

BM25 also uses inverse document frequency, but its term-frequency contribution differs from the TF-IDF definition above.

A common BM25 scoring formula is:

$$
score(q,d)=\sum_{t\in q} IDF(t)\,
\frac{f(t,d)(k_1+1)}
{f(t,d)+k_1\left(1-b+b\frac{|d|}{avgdl}\right)}
$$

with IDF:

$$
IDF(t)=\log\left(1+\frac{N-df(t)+0.5}{df(t)+0.5}\right)
$$

Parameters:
- $k_1$ controls how quickly term-frequency gains saturate. A common starting value is 1.5.
- $b$ controls document-length normalization. A common starting value is 0.75.
- $|d|$ is the document's token count and \(avgdl\) is the average document length.

### The crucial difference

**TF-IDF in this notebook:** term frequency is divided by document length, then the vectors are compared using cosine similarity.

**BM25:** each query term receives a nonlinear, saturating TF contribution. Document length is handled explicitly relative to average length. BM25 does not need cosine similarity.

As term frequency grows, BM25's TF contribution increases but with diminishing returns. Doubling a term's count therefore does not necessarily double its contribution.

In [ ]:
def bm25_idf(term, df, n_docs):
    """BM25's commonly used positive IDF variant."""
    term_df = df.get(term, 0)
    return math.log(1 + (n_docs - term_df + 0.5) / (term_df + 0.5))

def bm25_term_contribution(term_frequency, doc_length, avg_doc_length,
                           k1=1.5, b=0.75):
    """BM25 TF saturation and document-length normalization for one term."""
    if term_frequency == 0:
        return 0.0
    length_norm = 1 - b + b * (doc_length / avg_doc_length)
    return (
        term_frequency * (k1 + 1)
        / (term_frequency + k1 * length_norm)
    )

def bm25_score(query_tokens, doc_tokens, df, n_docs, avg_doc_length,
               k1=1.5, b=0.75):
    """Calculate BM25 score. Repeated query terms are counted as repeated terms."""
    counts = Counter(doc_tokens)
    doc_length = len(doc_tokens)
    score = 0.0
    for term in query_tokens:
        tf = counts[term]
        score += bm25_idf(term, df, n_docs) * bm25_term_contribution(
            tf, doc_length, avg_doc_length, k1=k1, b=b
        )
    return score

def rank_bm25(query_tokens, tokenized_docs, k1=1.5, b=0.75):
    """Rank documents using BM25 with corpus-derived statistics."""
    n_docs = len(tokenized_docs)
    df = document_frequency(tokenized_docs)
    avgdl = sum(len(tokens) for tokens in tokenized_docs.values()) / n_docs
    scored = [
        (doc_id, bm25_score(query_tokens, tokens, df, n_docs, avgdl, k1, b))
        for doc_id, tokens in tokenized_docs.items()
    ]
    return sorted(scored, key=lambda item: (-item[1], item[0]))

bm25_ranking = rank_bm25(query_tokens, tokenized_docs)
print("Average document length:", round(
    sum(len(t) for t in tokenized_docs.values()) / len(tokenized_docs), 2
))
print("\\nBM25 ranking:")
for doc_id, score in bm25_ranking:
    print(f"{doc_id}: {score:.4f}")

## 6. Inspect BM25 term-frequency saturation

Keep document length and IDF aside for a moment and look only at the BM25 TF contribution. The contribution rises quickly at first, then gains become smaller.

This is the core contrast with a simple linear TF term. BM25's exact TF contribution also depends on document length through `length_norm`.

In [ ]:
for tf in range(1, 11):
    contribution = bm25_term_contribution(
        term_frequency=tf,
        doc_length=10,
        avg_doc_length=10,
        k1=1.5,
        b=0.75
    )
    print(f"tf={tf:2d} -> BM25 TF contribution={contribution:.4f}")

### Compare length normalization

The next cell holds term frequency fixed while changing document length. When a document is longer than average, BM25 reduces the contribution of the same matching term; shorter-than-average documents receive less length penalty.

In [ ]:
avgdl = sum(len(t) for t in tokenized_docs.values()) / len(tokenized_docs)
fixed_tf = 2

for length in [max(1, int(avgdl * 0.5)), int(avgdl), int(avgdl * 2)]:
    contribution = bm25_term_contribution(
        term_frequency=fixed_tf,
        doc_length=length,
        avg_doc_length=avgdl,
        k1=1.5,
        b=0.75
    )
    print(f"doc_length={length:2d}, avgdl={avgdl:.2f} -> contribution={contribution:.4f}")

## 7. Compare all rankings side by side

A score's numeric scale is **not comparable across algorithms**. Compare document order within each method, and inspect the underlying formula to explain why.

In [ ]:
def ranking_positions(ranking):
    return {doc_id: rank for rank, (doc_id, _) in enumerate(ranking, start=1)}

rankings = {
    "Raw lexical": lexical_ranking,
    "TF-IDF cosine": tfidf_ranking,
    "BM25": bm25_ranking,
}

positions = {name: ranking_positions(ranking)
             for name, ranking in rankings.items()}

print(f"{'Document':<10} {'Lexical score':>14} {'TF-IDF score':>14} {'BM25 score':>12} "
      f"{'Lex rank':>9} {'TF-IDF rank':>12} {'BM25 rank':>9}")
print("-" * 100)

lex_scores = dict(lexical_ranking)
tfidf_scores = dict(tfidf_ranking)
bm25_scores = dict(bm25_ranking)

for doc_id in documents:
    print(f"{doc_id:<10} {lex_scores[doc_id]:>14.4f} "
          f"{tfidf_scores[doc_id]:>14.4f} {bm25_scores[doc_id]:>12.4f} "
          f"{positions['Raw lexical'][doc_id]:>9} "
          f"{positions['TF-IDF cosine'][doc_id]:>12} "
          f"{positions['BM25'][doc_id]:>9}")

## 8. Controlled experiment: change BM25 parameters

BM25's `k1` and `b` are meaningful parameters, not arbitrary decorations:
- Lower `k1` means term-frequency gains saturate sooner.
- Higher `k1` lets term frequency influence scores for longer.
- `b=0` disables document-length normalization.
- `b=1` applies full normalization relative to average document length.

Try the settings below to see whether the ranking changes.

In [ ]:
settings = [
    ("Low saturation, no length norm", 0.2, 0.0),
    ("Default-style settings", 1.5, 0.75),
    ("Higher k1, stronger length norm", 2.0, 1.0),
]

for label, k1, b in settings:
    ranking = rank_bm25(query_tokens, tokenized_docs, k1=k1, b=b)
    order = " > ".join(doc_id for doc_id, _ in ranking)
    print(f"{label} (k1={k1}, b={b}): {order}")

## 9. Final takeaways

| Property | Raw lexical baseline | TF-IDF + cosine | BM25 |
|---|---|---|---|
| Exact term matching | Yes | Yes | Yes |
| Rare terms weighted more | No | Yes, through IDF | Yes, through IDF |
| Term frequency used | Not in this baseline | Yes | Yes, with saturation |
| Document length handled | No | Indirectly through normalized TF and cosine | Explicitly using document length / average length |
| Vector cosine similarity required | No | Yes in this implementation | No |
| Main scoring idea | Count matching terms | Compare weighted vectors | Sum IDF-weighted, saturated term contributions |

### One-sentence explanation for students

**TF-IDF asks how important the terms are in a document and compares weighted vectors; BM25 scores each query-term match with diminishing returns for repetition and an explicit length adjustment.**

### Caveats
- Real search systems may use tokenization, stemming, stop-word rules, field boosts, and query operators that are absent from this toy example.
- TF-IDF is a family of variants; a different TF or IDF definition can change rankings.
- BM25 implementations can differ in IDF variant and query-term-frequency handling. This notebook uses a common positive IDF formula and sums contributions for each query-token occurrence.
- A ranking difference is not guaranteed for every corpus or query. The methods can agree on simple examples and diverge on others.